<a href="https://colab.research.google.com/github/piaseckazaneta/Python/blob/main/Co_naprawd%C4%99_decyduje_o_odej%C5%9Bciu_z_firmy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import duckdb

In [104]:
pd.set_option('display.max_columns', None)

In [4]:
ogolne = pd.read_csv("0.df_ogolne.csv")
zespol = pd.read_csv("1.df_zespol.csv")
tech = pd.read_csv("2.df_technologie.csv")
projekty = pd.read_csv("3.df_projekty.csv")
org_pracy = pd.read_csv("4.df_organizacja_pracy.csv")
well_being = pd.read_csv("5.df_well_being.csv")
przyszlosc = pd.read_csv("6.df_przyszlosc.csv")

tables = [ogolne, zespol, tech, projekty, org_pracy, well_being, przyszlosc]

In [5]:
org_pracy.head()

,id,w jaki sposób znalazłeś/aś obecną pracę,w jakiej formie jesteś obecnie zatrudniony/a,w jakim modelu pracujesz na co dzień,jak chciał(a)byś pracować na co dzień,zarobki,jak oceniasz konkurencyjność swojego wynagrodzenia względem rynku,kiedy ostatni raz otrzymałeś/aś podwyżkę wynagrodzenia,jak często w twojej firmie odbywają się rozmowy dotyczące wynagrodzenia lub przeglądy płacowe,czy w twojej firmie funkcjonują premie uznaniowe lub za wyniki,...,benefity | dofinansowanie posiłków / karta lunchowa,benefity | dofinansowanie zdrowia fizycznego lub psychicznego,benefity | dostęp do platform edukacyjnych,benefity | mentoring / programy rozwojowe,benefity | możliwość pracy z dowolnego miejsca,benefity | dodatkowe dni wolne,benefity | opcje na udziały / akcje w firmie,benefity | spotkania zespołowe / wyjazdy integracyjne,benefity | budżet na home office / sprzęt do pracy zdalnej,benefity | dni wolne od spotkań
0,0,awans wewnętrzny / zmiana stanowiska w tej sam...,b2b (samozatrudnienie),hybrydowo - większość z biura,hybrydowo - większość z domu,11000.0,poniżej średniej,ponad rok temu,"regularnie – co 6 miesięcy, co rok",nie – brak systemu premiowego,...,0,0,1,1,1,0,0,1,0,0
1,1,"portal z ofertami pracy (np. no fluff jobs, ju...",umowa o pracę,hybrydowo - większość z domu,zdalnie,7500.0,na poziomie średniej rynkowej,nigdy nie otrzymałem/am podwyżki w obecnej firmie,tylko na moją inicjatywę,tak – regularne premie (np. roczne),...,0,0,0,0,0,0,0,0,0,0
2,2,awans wewnętrzny / zmiana stanowiska w tej sam...,umowa o pracę,hybrydowo - większość z domu,zdalnie,10000.0,powyżej średniej,w ciągu ostatniego roku,"regularnie – co 6 miesięcy, co rok",tak – regularne premie (np. roczne),...,0,1,0,0,0,1,0,1,0,0
3,3,"portal z ofertami pracy (np. no fluff jobs, ju...",umowa o pracę,stacjonarnie,zdalnie,10000.0,poniżej średniej,ponad rok temu,nieregularnie – zależy od sytuacji lub wyników,tak – regularne premie (np. roczne),...,1,0,0,1,0,1,0,1,0,0
4,4,"portal z ofertami pracy (np. no fluff jobs, ju...",umowa o pracę,hybrydowo - większość z domu,hybrydowo - większość z biura,9800.0,na poziomie średniej rynkowej,w ciągu ostatniego roku,"regularnie – co 6 miesięcy, co rok",tak – regularne premie (np. roczne),...,0,0,0,0,1,0,0,0,0,0


In [6]:
for table in tables:
  print(table.shape)
  print(table.columns)
  print(table.head(2))


(297, 22)
Index(['id', 'stanowisko', 'płeć', 'miejsce zamieszkania',
       'kierunek wykształcenia', 'lata doświadczenia',
       'poziom stanowiska (seniority)', 'ile lat pracujesz na tym stanowisku',
       'wymiar pracy', 'wielkość firmy, w której pracujesz',
       'w jakim modelu realizujesz swoje projekty',
       'jaki kapitał ma firma, w której pracujesz',
       'czy przed objęciem obecnego stanowiska pracowałeś/aś już w branży it lub analitycznej',
       'wykształcenie__studiowała/em, ale nie ukończyła/em',
       'wykształcenie__doktorat', 'wykształcenie__inżynier',
       'wykształcenie__licencjat', 'wykształcenie__magister',
       'wykształcenie__studia podyplomowe', 'wykształcenie__zawodowe',
       'wykształcenie__średnie', 'rodzic'],
      dtype='object')
   id    stanowisko       płeć                        miejsce zamieszkania  \
0   0  data analyst  mężczyzna  duże miasto (powyżej 500 tys. mieszkańców)   
1   1  data analyst    kobieta  duże miasto (powyżej 500 ty

In [7]:
ogolne['w jakim modelu realizujesz swoje projekty'].value_counts()

,count
w jakim modelu realizujesz swoje projekty,
"jestem zatrudniony/a bezpośrednio w firmie, w której realizuję projekty",255
pracuję w software house / konsultingu – realizuję projekty dla różnych klientów,20
"jestem zatrudniony/a przez inną firmę (np. outsourcing, body leasing)",18
inne / trudno powiedzieć,4


Kto średnio więcej zarabia?

In [8]:
plec_zarobki = ogolne[['id', 'płeć']].merge(
    org_pracy[['id','zarobki']],
    on='id',
    how='inner'
)

In [9]:
plec_zarobki.groupby('płeć')['zarobki'].mean()

,zarobki
płeć,
kobieta,9179.395973
mężczyzna,9804.491525


Jakie stanowisko ma najwyższy odsetek osób z wykształceniem informatycznym?

In [10]:
czy_informatyczne = ogolne['kierunek wykształcenia'] == 'Informatyczne'


In [11]:
odsetek = czy_informatyczne.groupby(ogolne['stanowisko']).mean()

In [12]:
odsetek

,kierunek wykształcenia
stanowisko,
analytics engineer,0.250000
business analyst,0.100000
business intelligence (bi) developer,0.346154
data analyst,0.191358
data engineer,0.242424
data scientist,0.230769
ml/ai engineer,0.461538


Czy mediana zarobków osób pracujących 9-10 godzin jest wyższa, niż osob, które pracują 8 godzin?

In [13]:
czas_pracy_zarobki = well_being[['id','ile godzin dziennie średnio pracujesz']].merge(
    org_pracy[['id','zarobki']],
    on='id',
    how='inner'
)

In [14]:
czas_pracy_zarobki.groupby('ile godzin dziennie średnio pracujesz')['zarobki'].median().sort_values(ascending=False)

,zarobki
ile godzin dziennie średnio pracujesz,
9–10 godzin,9600.0
więcej,9500.0
6–7 godzin,8450.0
8 godzin,8200.0
mniej niż 6 godzin,8000.0


In [15]:
przyszlosc['jak oceniasz przyszłość branży data / it'].mean().round(2)

np.float64(6.29)

Czy osoby zatrudnione na B2B średnio więcej godzin spędzają na programowaniu (w porównaniu do pozostałych form zatrudnienia)?

In [16]:
org_pracy[('w jakiej formie jesteś obecnie zatrudniony/a')].value_counts()

,count
w jakiej formie jesteś obecnie zatrudniony/a,
umowa o pracę,243
b2b (samozatrudnienie),39
umowa zlecenie / o dzieło,6
inne,5


In [17]:
wynik = org_pracy[['id', 'w jakiej formie jesteś obecnie zatrudniony/a']].merge(
    tech[['id', 'jaki procent twojej pracy stanowi programowanie']],
    on='id',
    how='left').merge(well_being[['id', 'ile godzin dziennie średnio pracujesz']],
       on='id',
       how='left')

In [18]:
wynik['ile godzin dziennie średnio pracujesz'].unique()

array(['8 godzin', '9–10 godzin', 'mniej niż 6 godzin', '6–7 godzin', nan,
       'więcej'], dtype=object)

In [19]:
mapa_godzin = {
    '8 godzin': 8,
    '9-10 godzin': 9.5,
    '6-7 godzin': 6.5,
    'mniej niż 6 godzin': 5,
    'więcej': 11,
    'nan': np.nan
}

In [20]:
wynik['godziny_dziennie'] = wynik['ile godzin dziennie średnio pracujesz'].map(mapa_godzin)
print(wynik['godziny_dziennie'])

0      8.0
1      8.0
2      8.0
3      NaN
4      5.0
      ... 
292    8.0
293    8.0
294    5.0
295    NaN
296    8.0
Name: godziny_dziennie, Length: 297, dtype: float64


In [21]:
wynik['jaki procent twojej pracy stanowi programowanie'].describe()

,jaki procent twojej pracy stanowi programowanie
count,295.000000
mean,5.342373
std,2.413412
min,1.000000
25%,3.500000
50%,6.000000
75%,7.000000
max,10.000000


In [22]:
wynik['czas_programowania'] = wynik['godziny_dziennie'] * wynik['jaki procent twojej pracy stanowi programowanie'] / 10

In [23]:
programowanie = wynik.groupby('w jakiej formie jesteś obecnie zatrudniony/a')['czas_programowania'].mean().round(2)

In [24]:
programowanie

,czas_programowania
w jakiej formie jesteś obecnie zatrudniony/a,
b2b (samozatrudnienie),4.95
inne,4.27
umowa o pracę,4.23
umowa zlecenie / o dzieło,3.30


Jakie stanowisko ma najmniejszy odsetek osób, które mają wpływ na wybór technologii (zaznaczyło "tak" w pytaniu "czy masz wpływ na wybór technologii (tech stacku) w swoim zespole / firmie")

In [25]:
stanowisko_tech = ogolne[['id', 'stanowisko']].merge(
    tech[['id', 'czy masz wpływ na wybór technologii (tech stacku) w swoim zespole / firmie']],
    on='id',
    how='left'
)

In [26]:
stanowisko_tech['wplyw'] = stanowisko_tech['czy masz wpływ na wybór technologii (tech stacku) w swoim zespole / firmie'].map({'tak': 1, 'nie': 0})

In [27]:
wybor_tech = stanowisko_tech.groupby('stanowisko')['wplyw'].mean().round(2)

In [28]:
wybor_tech

,wplyw
stanowisko,
analytics engineer,1.00
business analyst,0.40
business intelligence (bi) developer,0.35
data analyst,0.36
data engineer,0.13
data scientist,0.61
ml/ai engineer,0.80


Która z poniższych grup średnio najlepiej ocenia swój zespół pod względem relacji personalnych?

In [29]:
zespol['jaki jest przybliżony rozkład płci w twoim zespole']

,jaki jest przybliżony rozkład płci w twoim zespole
0,zrównoważony pod względem płci
1,w większości męski
2,w większości żeński
3,w większości męski
4,w większości żeński
...,...
292,zrównoważony pod względem płci
293,w większości żeński
294,zrównoważony pod względem płci
295,w większości żeński


In [30]:
zespol['jak oceniasz swój zespół pod względem relacji personalnych (atmosfera, współpraca)']

,"jak oceniasz swój zespół pod względem relacji personalnych (atmosfera, współpraca)"
0,10.0
1,9.0
2,7.0
3,10.0
4,8.0
...,...
292,6.0
293,7.0
294,7.0
295,9.0


In [31]:
plec_ocena_relacji = ogolne[['id', 'płeć']].merge(
    zespol[['id', 'jak oceniasz swój zespół pod względem relacji personalnych (atmosfera, współpraca)',
            'jaki jest przybliżony rozkład płci w twoim zespole']],
    on='id',
    how='left'
)

In [32]:
ocena = plec_ocena_relacji.groupby(['płeć','jaki jest przybliżony rozkład płci w twoim zespole'])['jak oceniasz swój zespół pod względem relacji personalnych (atmosfera, współpraca)'].mean().round(2).sort_values(ascending=False)

In [33]:
ocena

płeć       jaki jest przybliżony rozkład płci w twoim zespole
mężczyzna  zrównoważony pod względem płci                        8.02
kobieta    w większości żeński                                   7.95
mężczyzna  w większości żeński                                   7.85
           w większości męski                                    7.81
kobieta    w większości męski                                    7.69
           zrównoważony pod względem płci                        7.68
Name: jak oceniasz swój zespół pod względem relacji personalnych (atmosfera, współpraca), dtype: float64

Jak praca na projektach wpływa na zadowolenie z pracy? Która grupa osób jest średnio najbardziej zadowolona?

In [34]:
projekty['jak często pracujesz w formie projektów'].value_counts()

,count
jak często pracujesz w formie projektów,
"częściowo – część zadań to projekty, część bieżące zadania",126
regularnie – większość mojej pracy ma charakter projektowy,110
rzadko – głównie bieżące zadania,38
w ogóle – moja praca nie ma charakteru projektowego,20


In [35]:
well_being['w jaki stopniu jesteś zadowolona/y ze swojej pracy']

,w jaki stopniu jesteś zadowolona/y ze swojej pracy
0,9.0
1,5.0
2,7.0
3,7.0
4,2.0
...,...
292,4.0
293,5.0
294,3.0
295,5.0


In [36]:
zadowolenie_projekty = projekty[['id', 'jak często pracujesz w formie projektów']].merge(
    well_being[['id', 'w jaki stopniu jesteś zadowolona/y ze swojej pracy']],
    on='id',
    how='left'
)

In [37]:
ocena_projekty = zadowolenie_projekty.groupby('jak często pracujesz w formie projektów')['w jaki stopniu jesteś zadowolona/y ze swojej pracy'].mean().round(2).sort_values(ascending=False)
print(ocena_projekty)

jak często pracujesz w formie projektów
regularnie – większość mojej pracy ma charakter projektowy    7.07
częściowo – część zadań to projekty, część bieżące zadania    7.01
w ogóle – moja praca nie ma charakteru projektowego           5.95
rzadko – głównie bieżące zadania                              5.92
Name: w jaki stopniu jesteś zadowolona/y ze swojej pracy, dtype: float64


Czy rodzice częściej niż osoby nieposiadające dzieci zaznaczali odpowiedź „8 godzin” jako dzienny czas pracy?

In [44]:
rodzice_czas_pracy = ogolne[['id', 'rodzic']].merge(
    well_being[['id', 'ile godzin dziennie średnio pracujesz']],
    on='id',
    how='left'
)

In [46]:
rodzic_praca = rodzice_czas_pracy.groupby('rodzic')['ile godzin dziennie średnio pracujesz'].value_counts(normalize=True)
print(rodzic_praca)

#normaliza=True liczy % i wynik wychodzi prawidłowy bo liczba rodzicow i nierodzicow jest rozna, wiec odsetek musi byc

rodzic  ile godzin dziennie średnio pracujesz
False   8 godzin                                 0.576190
        6–7 godzin                               0.180952
        9–10 godzin                              0.114286
        mniej niż 6 godzin                       0.095238
        więcej                                   0.033333
True    8 godzin                                 0.698795
        6–7 godzin                               0.156627
        9–10 godzin                              0.084337
        mniej niż 6 godzin                       0.060241
Name: proportion, dtype: float64


Czy osoby, które zaznaczyły, że się regularnie szkolą, średnio lepiej oceniają przyszłość branży data / it?

In [47]:
przyszlosc['jak oceniasz przyszłość branży data / it'].value_counts()

,count
jak oceniasz przyszłość branży data / it,
7.0,68
8.0,53
5.0,51
6.0,49
4.0,25
10.0,12
9.0,11
3.0,9
2.0,7


In [50]:
przyszlosc.groupby('czy obecnie szkolisz się lub rozwijasz swoje umiejętności zawodowe')['jak oceniasz przyszłość branży data / it'].mean()

,jak oceniasz przyszłość branży data / it
czy obecnie szkolisz się lub rozwijasz swoje umiejętności zawodowe,
"nie, ale planuję w najbliższym czasie",6.333333
"nie, na ten moment skupiam się na innych aspektach życia",6.333333
okazjonalnie,6.120879
"tak, regularnie",6.379747


Wybierz 3 najczęściej wybierane przyczyny zadowolenia z pracy:

In [62]:
kolumny_zadowolenie = [col for col in well_being.columns if 'zadowolenie' in col.lower()]
print(kolumny_zadowolenie)

['zadowolenie | możliwość rozwoju', 'zadowolenie | przełożony / sposób zarządzania', 'zadowolenie | stabilność zatrudnienia', 'zadowolenie | work-life balance', 'zadowolenie | wynagrodzenie', 'zadowolenie | zakres obowiązków / ciekawe projekty', 'zadowolenie | zespół i atmosfera']


In [64]:
wyniki_zadowolenie = well_being[kolumny_zadowolenie].sum().sort_values(ascending=False).head(3)
print(wyniki_zadowolenie)

zadowolenie | stabilność zatrudnienia    187
zadowolenie | zespół i atmosfera         186
zadowolenie | work-life balance          156
dtype: int64


Czy całkowite doświadczenie zawodowe wpływa na zarobki, nawet jeśli staż na danym stanowisku jest taki sam?

In [95]:
seniority_zarobki_staz = ogolne[['id', 'poziom stanowiska (seniority)', 'lata doświadczenia', 'ile lat pracujesz na tym stanowisku']].merge(
    org_pracy[['id', 'zarobki']],
    on='id',
    how='left'
)


In [102]:
wynik = seniority_zarobki_staz.groupby(['lata doświadczenia','poziom stanowiska (seniority)', 'ile lat pracujesz na tym stanowisku'])['zarobki'].mean().round(2)

In [106]:
print(wynik)

lata doświadczenia  poziom stanowiska (seniority)  ile lat pracujesz na tym stanowisku
11-15 lat           junior                         1-2 lata                                6750.00
                                                   3-4 lata                                5000.00
                                                   do 1 roku                               5000.00
                    lead / expert                  3-4 lata                               10450.00
                                                   5 lub więcej                           15000.00
                                                                                            ...   
ponad 15 lat        mid / regular                  5 lub więcej                            6300.00
                                                   do 1 roku                              12233.33
                    senior                         1-2 lata                                    NaN
                      

In [89]:
wynik.groupby('lata doświadczenia')['zarobki'].mean().round(2)

KeyError: 'lata doświadczenia'

In [69]:
sen_zarobki_staz.groupby(['lata doświadczenia', 'ile lat pracujesz na tym stanowisku'])['zarobki'].mean().round(2)

lata doświadczenia  ile lat pracujesz na tym stanowisku
11-15 lat           1-2 lata                                9856.25
                    3-4 lata                                8919.23
                    5 lub więcej                           12130.00
                    do 1 roku                              10885.71
3–5 lat             1-2 lata                                8236.54
                    3-4 lata                                8352.63
                    5 lub więcej                            7200.00
                    do 1 roku                               8210.61
6–10 lat            1-2 lata                               11092.58
                    3-4 lata                                9804.76
                    5 lub więcej                           11942.86
                    do 1 roku                               9914.00
do 2 lat            1-2 lata                                7816.25
                    do 1 roku                               5700.00
ponad 15 lat        1-2 lata                                6750.00
                    3-4 lata                               10871.43
                    5 lub więcej                            9550.00
                    do 1 roku                              12540.00
Name: zarobki, dtype: float64

In [38]:
# Inicjalizacja DuckDB w pamięci
con = duckdb.connect()

In [39]:
# Rejestracja DataFrame'ów w silniku DuckDB
con.register('ogolne', ogolne)
con.register('zespol', zespol)
con.register('tech', tech)
con.register('projekty', projekty)
con.register('org_pracy', org_pracy)
con.register('well_being', well_being)
con.register('przyszlosc', przyszlosc)

In [40]:
query = """
CREATE OR REPLACE TABLE master_data AS
SELECT *
FROM ogolne
LEFT JOIN projekty USING (id)
LEFT JOIN well_being USING (id)
LEFT JOIN zespol USING (id)
LEFT JOIN przyszlosc USING (id)
LEFT JOIN tech USING (id)
LEFT JOIN org_pracy USING (id);
"""

con.execute(query)
df_master = con.execute("SELECT * FROM master_data").df()
print(f"Kształt: {df_master.shape}")

Kształt: (297, 169)


In [41]:
df_master.head()

,id,stanowisko,płeć,miejsce zamieszkania,kierunek wykształcenia,lata doświadczenia,poziom stanowiska (seniority),ile lat pracujesz na tym stanowisku,wymiar pracy,"wielkość firmy, w której pracujesz",...,benefity | dofinansowanie posiłków / karta lunchowa,benefity | dofinansowanie zdrowia fizycznego lub psychicznego,benefity | dostęp do platform edukacyjnych,benefity | mentoring / programy rozwojowe,benefity | możliwość pracy z dowolnego miejsca,benefity | dodatkowe dni wolne,benefity | opcje na udziały / akcje w firmie,benefity | spotkania zespołowe / wyjazdy integracyjne,benefity | budżet na home office / sprzęt do pracy zdalnej,benefity | dni wolne od spotkań
0,0,data analyst,mężczyzna,duże miasto (powyżej 500 tys. mieszkańców),"Inne techniczne (np. inżynieria, automatyka, f...",11-15 lat,mid / regular,do 1 roku,pełny etat (ok. 40 godzin tygodniowo),duża firma (powyżej 500 pracowników),...,0,0,1,1,1,0,0,1,0,0
1,1,data analyst,kobieta,duże miasto (powyżej 500 tys. mieszkańców),"Inne techniczne (np. inżynieria, automatyka, f...",6–10 lat,mid / regular,do 1 roku,pełny etat (ok. 40 godzin tygodniowo),duża firma (powyżej 500 pracowników),...,0,0,0,0,0,0,0,0,0,0
2,2,data engineer,kobieta,duże miasto (powyżej 500 tys. mieszkańców),Ekonomiczne / finansowe,6–10 lat,junior,1-2 lata,pełny etat (ok. 40 godzin tygodniowo),duża firma (powyżej 500 pracowników),...,0,1,0,0,0,1,0,1,0,0
3,3,data analyst,mężczyzna,małe miasto (poniżej 100 tys. mieszkańców),Informatyczne,11-15 lat,manager / team leader,3-4 lata,pełny etat (ok. 40 godzin tygodniowo),duża firma (powyżej 500 pracowników),...,1,0,0,1,0,1,0,1,0,0
4,4,business analyst,kobieta,duże miasto (powyżej 500 tys. mieszkańców),Ekonomiczne / finansowe,3–5 lat,mid / regular,3-4 lata,pełny etat (ok. 40 godzin tygodniowo),duża firma (powyżej 500 pracowników),...,0,0,0,0,1,0,0,0,0,0


In [42]:
# 1. Zmienna kluczowa: Ile osób rozważa odejście z pracy?
col_odejscie = 'czy obecnie rozważasz zmianę pracy'
print("--- ROZKŁAD DECYZJI O ZMIANIE PRACY ---")
print(df_master[col_odejscie].value_counts(dropna=False))
print("\nProcentowo:")
print(df_master[col_odejscie].value_counts(normalize=True).round(3) * 100)

# 2. Zarobki: Podstawowe statystyki opisowe
col_zarobki = 'zarobki'
print("\n--- STATYSTYKI ZAROBKÓW ---")
print(df_master[col_zarobki].describe().round(2))

# 3. Braki danych w kluczowych metrykach
print("\n--- BRAKI DANYCH W WYBRANYCH KOLUMNACH ---")
print(df_master[[col_odejscie, col_zarobki, 'w jaki stopniu jesteś zadowolona/y ze swojej pracy']].isna().sum())

--- ROZKŁAD DECYZJI O ZMIANIE PRACY ---
czy obecnie rozważasz zmianę pracy
tak – rozglądam się, ale bez presji    124
nie                                    123
tak – aktywnie szukam nowej pracy       25
trudno powiedzieć                       22
None                                     3
Name: count, dtype: int64

Procentowo:
czy obecnie rozważasz zmianę pracy
tak – rozglądam się, ale bez presji    42.2
nie                                    41.8
tak – aktywnie szukam nowej pracy       8.5
trudno powiedzieć                       7.5
Name: proportion, dtype: float64

--- STATYSTYKI ZAROBKÓW ---
count      267.00
mean      9455.66
std       4033.32
min       3834.00
25%       6700.00
50%       8500.00
75%      11000.00
max      31000.00
Name: zarobki, dtype: float64

--- BRAKI DANYCH W WYBRANYCH KOLUMNACH ---
czy obecnie rozważasz zmianę pracy                     3
zarobki                                               30
w jaki stopniu jesteś zadowolona/y ze swojej pracy     3
dtype: in

In [43]:
ogolne['rodzic'].value_counts()

,count
rodzic,
False,213
True,84
